# LEAP Python Workshop: Which Marketing Channel Actually Works?

A store ran ads on two channels. Your job: figure out which one to keep paying for.

**`channel_test.csv`** — one row per website visitor:

| Column | Meaning |
|---|---|
| `visitor_id` | an ID for each person |
| `channel` | `Paid Social` or `Retargeting` |
| `visitor_type` | `new` or `returning` |
| `converted` | `1` if they bought, `0` if not |

**`channel_spend.csv`** — how much was spent on each channel. Two rows.

4 exercises. Run a cell with **Shift + Enter**.
Keep both CSVs in the same folder as this notebook (on Colab, upload them first).

## Setup

`pandas` is the standard Python library for tables. `read_csv` loads a file into a
**DataFrame** — basically a spreadsheet inside Python. `visitors` and `spend` are
**variables**: names holding a value. Anything after `#` is a comment.

In [ ]:
import pandas as pd

visitors = pd.read_csv("channel_test.csv")
spend = pd.read_csv("channel_spend.csv")

---
## Exercise 1: Look at the data

| What you want | How |
|---|---|
| first 5 rows | `visitors.head()` |
| number of rows | `len(visitors)` |
| count each value | `visitors["channel"].value_counts()` |
| add up a column | `visitors["converted"].sum()` |
| average a column | `visitors["converted"].mean()` |

The last line of a cell displays on its own — you only need `print()` to show two
things at once.

**Tasks**

1. Show the first few rows of `visitors`, and the whole `spend` table.
2. How many visitors came from each channel?
3. How many purchases in total?

In [ ]:
# --- Exercise 1 ---
print(visitors.head())
print(spend)
print(visitors["channel"].value_counts())
print("total purchases:", visitors["converted"].sum())

---
## Exercise 2: Conversion rate by channel

**Filter** rows with a condition in square brackets — the Excel filter button, in code:

```python
social = visitors[visitors["channel"] == "Paid Social"]
```

`=` assigns a value, `==` asks "are these equal?"

**The 0/1 trick:** `converted` is only 0s and 1s, so its *average* is the conversion
rate. `social["converted"].mean()` → `0.074`, meaning 7.4%.

**Tasks**

1. Get the conversion rate for Paid Social.
2. Get it for Retargeting.
3. Which looks better?
4. You wrote that math twice, so turn it into a **function** and check it matches:

```python
def rate_of(table):
    return table["converted"].mean()
```

In [ ]:
# --- Exercise 2 ---
social = visitors[visitors["channel"] == "Paid Social"]
retarget = visitors[visitors["channel"] == "Retargeting"]

social_rate = social["converted"].mean()
retarget_rate = retarget["converted"].mean()

print("Paid Social:", social_rate)     # 7.4%
print("Retargeting:", retarget_rate)   # 12.0%
# Retargeting wins. Case closed? See Exercise 3.

In [ ]:
def rate_of(table):
    return table["converted"].mean()


rate_of(social)   # matches above

---
## Exercise 3: The plot twist

The channels weren't shown to the same kind of people. To split a group, filter a table
you already filtered:

```python
social_new = social[social["visitor_type"] == "new"]
```

**Tasks**

1. Make four variables: each channel split into new and returning.
2. Get all four rates with `rate_of()`.
3. Compare the channels among new visitors only, then among returning only. Does
   Exercise 2 still hold?
4. Run `len(social_new) / len(social)` for each channel. What happened?

In [ ]:
# --- Exercise 3 ---
social_new = social[social["visitor_type"] == "new"]
social_returning = social[social["visitor_type"] == "returning"]
retarget_new = retarget[retarget["visitor_type"] == "new"]
retarget_returning = retarget[retarget["visitor_type"] == "returning"]

# Paid Social first, Retargeting second
print("new      ", rate_of(social_new), rate_of(retarget_new))            # 6%  vs 4%
print("returning", rate_of(social_returning), rate_of(retarget_returning))  # 20% vs 14%
# Paid Social wins BOTH groups, yet looked worse overall.

In [ ]:
print("Paid Social, share new:", len(social_new) / len(social))
print("Retargeting, share new:", len(retarget_new) / len(retarget))

# Retargeting went almost entirely to returning visitors, who were already likely to
# buy. Paid Social was mostly reaching strangers. Retargeting didn't convert better --
# it got easier customers.
#
# This is Simpson's paradox: a trend that flips when you split the data into groups.

---
## Exercise 4: Chart it

A **list** is values in square brackets. `plt.bar()` takes labels and heights — same
order, same length.

```python
import matplotlib.pyplot as plt

plt.figure(figsize=(5, 4))
plt.bar(["Paid Social", "Retargeting"], [0.074, 0.12])
plt.ylabel("Conversion rate")
plt.title("Conversion rate by channel")
plt.show()
```

**Tasks**

1. Chart the two rates from Exercise 2.
2. Chart the four rates from Exercise 3, labeled like `"Paid Social - new"`. Add
   `plt.xticks(rotation=45)` so labels don't overlap.
3. Which chart would you show your boss? Which one is true?

In [ ]:
# --- Exercise 4 ---
import matplotlib.pyplot as plt

plt.figure(figsize=(5, 4))
plt.bar(["Paid Social", "Retargeting"], [social_rate, retarget_rate])
plt.ylabel("Conversion rate")
plt.title("Overall -- Retargeting looks better")
plt.show()

In [ ]:
labels = [
    "Paid Social - new",
    "Paid Social - returning",
    "Retargeting - new",
    "Retargeting - returning",
]
rates = [
    rate_of(social_new),
    rate_of(social_returning),
    rate_of(retarget_new),
    rate_of(retarget_returning),
]

plt.figure(figsize=(6, 4))
plt.bar(labels, rates)
plt.ylabel("Conversion rate")
plt.title("Split by visitor type -- Paid Social wins both")
plt.xticks(rotation=45)
plt.show()

## Wrap-up

Retargeting looked better overall — 12% vs 7.4%. Then we split by visitor type:

| | Paid Social | Retargeting |
|---|---|---|
| New | 6% | 4% |
| Returning | 20% | 14% |

Paid Social won both groups. It only looked worse because Retargeting's ads went mostly to returning visitors, who were already likely to buy. It got easier customers, not better results.

That flip is called **Simpson's paradox**, and it shows up everywhere: a hospital looks worse because it takes the sickest patients, a course looks easy because only seniors take it.

Today you loaded a CSV, filtered it, averaged a 0/1 column to get a rate, wrote a function, and charted it. That's a real workflow.

> An average hides its groups. Before you trust a number, split it.

**Try at home:** compute cost per sale from `channel_spend.csv` (Paid Social spent $30,000, Retargeting $60,000), then look up pandas `groupby` — it does Exercise 3 in one line.
